# Candidate Dataset Tour

In this notebook I inspect the labeled candidate plans used for the downstream plan-correctness task. The goal is to make the JSONL files easy to understand before looking at models or feature matrices.

In [4]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent.parent

DATASET = ROOT / "data" / "correctness_dataset"
CANDIDATES = DATASET / "candidates"
FEATURES = DATASET / "features"
HEADS = ROOT / "models" / "correctness_heads"
RESULTS = ROOT / "results" / "analysis"

print(f"Repository root: {ROOT}")

Repository root: /home/bernardod/desktop/research/planfm-validity


In [5]:
def load_candidates(split):
    return pd.read_json(CANDIDATES / f"{split}.jsonl", lines=True)

splits = ["train", "validation", "test-interpolation", "test-extrapolation"]
frames = []
for split in splits:
    frame = load_candidates(split)
    frame["split"] = split
    frames.append(frame)

candidates = pd.concat(frames, ignore_index=True)
print(f"Rows: {len(candidates):,}")
print(f"Columns: {list(candidates.columns)}")
candidates.head()

Rows: 2,946
Columns: ['candidate_id', 'domain', 'split', 'problem', 'corruption_type', 'plan', 'plan_len', 'gold_plan_len', 'label_valid', 'label_executable', 'correctness_score']


,candidate_id,domain,split,problem,corruption_type,plan,plan_len,gold_plan_len,label_valid,label_executable,correctness_score
0,blocks::train::probBLOCKS-4-0::000::gold,blocks,train,probBLOCKS-4-0,gold,"[(pick-up b), (stack b a), (pick-up c), (stack...",6,6,1,1,1.000000
1,blocks::train::probBLOCKS-4-0::001::truncate,blocks,train,probBLOCKS-4-0,truncate,[],0,6,0,0,0.000000
2,blocks::train::probBLOCKS-4-0::002::delete,blocks,train,probBLOCKS-4-0,delete,"[(pick-up b), (stack b a), (stack c b), (pick-...",5,6,0,0,0.366667
3,blocks::train::probBLOCKS-4-0::003::swap,blocks,train,probBLOCKS-4-0,swap,"[(pick-up b), (stack b a), (pick-up c), (stack...",6,6,0,0,0.666667
4,blocks::train::probBLOCKS-4-0::004::replace,blocks,train,probBLOCKS-4-0,replace,"[(pick-up b), (stack b a), (pick-up c), (stack...",6,6,0,0,0.666667


In [6]:
summary = (
    candidates.groupby("split")
    .agg(total=("candidate_id", "count"), valid=("label_valid", "sum"), executable=("label_executable", "sum"))
)
summary["invalid"] = summary["total"] - summary["valid"]
summary["non_executable"] = summary["total"] - summary["executable"]
summary

,total,valid,executable,invalid,non_executable
split,,,,,
test-extrapolation,1365,273,544,1092,821
test-interpolation,260,52,98,208,162
train,1156,231,463,925,693
validation,165,33,68,132,97


In [4]:
by_domain = pd.crosstab(candidates["domain"], candidates["label_valid"], margins=True)
by_domain = by_domain.rename(columns={0: "invalid", 1: "valid"})
by_domain

label_valid,invalid,valid,All
domain,,,
blocks,140,35,175
gripper,100,25,125
logistics,172,43,215
visitall-from-everywhere,1945,486,2431
All,2357,589,2946


In [5]:
corruptions = (
    candidates.groupby(["split", "corruption_type"])
    .size()
    .unstack(fill_value=0)
    .sort_index()
)
corruptions

corruption_type,delete,gold,insert,repeat,replace,swap,truncate
split,,,,,,,
test-extrapolation,273,273,14,0,273,259,273
test-interpolation,46,52,13,1,48,48,52
train,221,232,15,0,231,226,231
validation,33,33,2,0,33,31,33


In [6]:
def show_plan(row):
    print(f"candidate_id: {row.candidate_id}")
    print(f"domain/problem: {row.domain} / {row.problem}")
    print(f"corruption_type: {row.corruption_type}")
    print(f"label_valid: {row.label_valid}; label_executable: {row.label_executable}")
    print(f"plan length: {row.plan_len}; gold length: {row.gold_plan_len}")
    print("plan:")
    for step, action in enumerate(row.plan, start=1):
        print(f"  {step:02d}. {action}")

example_gold = candidates.query("domain == 'blocks' and split == 'train' and corruption_type == 'gold'").iloc[0]
show_plan(example_gold)

candidate_id: blocks::train::probBLOCKS-4-0::000::gold
domain/problem: blocks / probBLOCKS-4-0
corruption_type: gold
label_valid: 1; label_executable: 1
plan length: 6; gold length: 6
plan:
  01. (pick-up b)
  02. (stack b a)
  03. (pick-up c)
  04. (stack c b)
  05. (pick-up d)
  06. (stack d c)


In [7]:
example_bad = candidates.query("domain == 'blocks' and split == 'train' and corruption_type != 'gold'").iloc[0]
show_plan(example_bad)

candidate_id: blocks::train::probBLOCKS-4-0::001::truncate
domain/problem: blocks / probBLOCKS-4-0
corruption_type: truncate
label_valid: 0; label_executable: 0
plan length: 0; gold length: 6
plan:


In [8]:
example_bad = candidates.query("domain == 'blocks' and split == 'train' and corruption_type == 'replace'").iloc[0]
show_plan(example_bad)

candidate_id: blocks::train::probBLOCKS-4-0::004::replace
domain/problem: blocks / probBLOCKS-4-0
corruption_type: replace
label_valid: 0; label_executable: 0
plan length: 6; gold length: 6
plan:
  01. (pick-up b)
  02. (stack b a)
  03. (pick-up c)
  04. (stack c b)
  05. (put-down d)
  06. (stack d c)


In [9]:
example_bad = candidates.query("domain == 'blocks' and split == 'train' and corruption_type == 'replace'").iloc[0]
show_plan(example_bad)

candidate_id: blocks::train::probBLOCKS-4-0::004::replace
domain/problem: blocks / probBLOCKS-4-0
corruption_type: replace
label_valid: 0; label_executable: 0
plan length: 6; gold length: 6
plan:
  01. (pick-up b)
  02. (stack b a)
  03. (pick-up c)
  04. (stack c b)
  05. (put-down d)
  06. (stack d c)
